In [9]:
%%writefile bfs.cpp

#include <iostream>
#include <vector>
#include <queue>
#include <omp.h>

using namespace std;

const int MAX = 100;

vector<int> graph[MAX];
bool visited[MAX];

// Parallel BFS for Graph
void parallelBFS(int start) {

    queue<int> q;

    visited[start] = true;
    q.push(start);

    while (!q.empty()) {

        int size = q.size();

        #pragma omp parallel for
        for (int i = 0; i < size; i++) {

            int curr;

            // Critical section for queue access
            #pragma omp critical
            {
                curr = q.front();
                q.pop();

                cout << curr << " ";
            }

            // Traverse adjacent vertices
            for (int j = 0; j < graph[curr].size(); j++) {

                int neighbor = graph[curr][j];

                if (!visited[neighbor]) {

                    #pragma omp critical
                    {
                        if (!visited[neighbor]) {
                            visited[neighbor] = true;
                            q.push(neighbor);
                        }
                    }
                }
            }
        }
    }
}

int main() {

    int vertices, edges;

    cout << "Enter number of vertices: ";
    cin >> vertices;

    cout << "Enter number of edges: ";
    cin >> edges;

    cout << "Enter edges (u v):\n";

    for (int i = 0; i < edges; i++) {

        int u, v;
        cin >> u >> v;

        graph[u].push_back(v);
        graph[v].push_back(u); // Undirected graph
    }

    int start;

    cout << "Enter starting vertex: ";
    cin >> start;

    cout << "\nParallel BFS Traversal: ";

    parallelBFS(start);

    return 0;
}

Overwriting bfs.cpp


In [10]:
!g++ -fopenmp bfs.cpp -o bfs

In [11]:
!./bfs

Enter number of vertices: 4
Enter number of edges: 4
Enter edges (u v):
 1 0
2 0
 1 2
 2 3
Enter starting vertex: 0

Parallel BFS Traversal: 0 1 2 3 